<a href="https://colab.research.google.com/github/Komal5578/DWM/blob/main/OLAP_my_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PIVOT TABLE / OLAP in Python (paper-ready)
**How to use:** only edit the cell marked ✏️ **DATA**. Type the table from the question paper there, then *Runtime → Run all*.
Every operation below (roll-up, drill-down, slice, dice, pivot) reuses the same `df`, so nothing else needs to change.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)

## ✏️ DATA: type the paper's table here (one list per row)

In [ ]:
columns = ['Year','Quarter','Region','Category','Product','Sales','Profit','Quantity']
data = [
 [2023,'Q1','East','Furniture','Chair',531,152,7],
 [2023,'Q1','East','Technology','Phone',866,53,9],
 [2023,'Q1','West','Furniture','Table',296,41,1],
 [2023,'Q1','West','Office','Pen',719,74,2],
 [2023,'Q2','East','Furniture','Chair',644,99,4],
 [2023,'Q2','East','Technology','Phone',292,54,1],
 [2023,'Q2','West','Furniture','Table',779,63,4],
 [2023,'Q2','West','Office','Pen',845,174,1],
 [2023,'Q3','East','Furniture','Chair',790,155,1],
 [2023,'Q3','East','Technology','Phone',426,26,3],
 [2023,'Q3','West','Furniture','Table',496,76,9],
 [2023,'Q3','West','Office','Pen',320,61,9],
 [2023,'Q4','East','Furniture','Chair',898,85,10],
 [2023,'Q4','East','Technology','Phone',784,164,6],
 [2023,'Q4','West','Furniture','Table',299,55,2],
 [2023,'Q4','West','Office','Pen',777,50,4],
 [2024,'Q1','East','Furniture','Chair',708,155,7],
 [2024,'Q1','East','Technology','Phone',521,86,8],
 [2024,'Q1','West','Furniture','Table',570,71,3],
 [2024,'Q1','West','Office','Pen',449,31,5],
 [2024,'Q2','East','Furniture','Chair',737,128,6],
 [2024,'Q2','East','Technology','Phone',659,80,2],
 [2024,'Q2','West','Furniture','Table',320,56,3],
 [2024,'Q2','West','Office','Pen',550,48,8],
 [2024,'Q3','East','Furniture','Chair',631,37,11],
 [2024,'Q3','East','Technology','Phone',279,67,10],
 [2024,'Q3','West','Furniture','Table',521,70,6],
 [2024,'Q3','West','Office','Pen',808,140,8],
 [2024,'Q4','East','Furniture','Chair',270,70,5],
 [2024,'Q4','East','Technology','Phone',685,153,2],
 [2024,'Q4','West','Furniture','Table',262,60,5],
 [2024,'Q4','West','Office','Pen',862,167,11],
]
df = pd.DataFrame(data, columns=columns)
df

,Year,Quarter,Region,Category,Product,Sales,Profit,Quantity
0,2023,Q1,East,Furniture,Chair,531,152,7
1,2023,Q1,East,Technology,Phone,866,53,9
2,2023,Q1,West,Furniture,Table,296,41,1
3,2023,Q1,West,Office,Pen,719,74,2
4,2023,Q2,East,Furniture,Chair,644,99,4
5,2023,Q2,East,Technology,Phone,292,54,1
6,2023,Q2,West,Furniture,Table,779,63,4
7,2023,Q2,West,Office,Pen,845,174,1
8,2023,Q3,East,Furniture,Chair,790,155,1
9,2023,Q3,East,Technology,Phone,426,26,3


### (Optional) If the paper gives a *Date* column instead of Year/Quarter
Remove the `#` and run. It builds the time hierarchy (Year → Quarter → Month) automatically.
Other options: load a CSV with `df = pd.read_csv('file.csv')`.

In [ ]:
# df['Date'] = pd.to_datetime(df['Date'])
# df['Year'] = df['Date'].dt.year
# df['Quarter'] = 'Q' + df['Date'].dt.quarter.astype(str)
# df['Month'] = df['Date'].dt.month_name()

## 1. Base cube (pivot table)
`index` = rows, `columns` = columns, `values` = measure, `aggfunc` = how to combine, `margins=True` adds totals.

In [ ]:
cube = pd.pivot_table(df, index=['Year','Quarter'], columns='Category',
                      values='Sales', aggfunc='sum', fill_value=0,
                      margins=True, margins_name='Total')
cube

Category       Furniture  Office  Technology  Total
Year  Quarter                                      
2023  Q1             827     719         866   2412
      Q2            1423     845         292   2560
      Q3            1286     320         426   2032
      Q4            1197     777         784   2758
2024  Q1            1278     449         521   2248
      Q2            1057     550         659   2266
      Q3            1152     808         279   2239
      Q4             532     862         685   2079
Total               8752    5330        4512  18594

## 2. Roll-up (go UP the hierarchy: Quarter → Year, less detail)
Just use fewer fields in `index`.

In [ ]:
rollup = pd.pivot_table(df, index='Year', columns='Category', values='Sales',
                        aggfunc='sum', fill_value=0, margins=True, margins_name='Total')
rollup

Category,Furniture,Office,Technology,Total
Year,,,,
2023,4733,2661,2368,9762
2024,4019,2669,2144,8832
Total,8752,5330,4512,18594


## 3. Drill-down (go DOWN: Year → Quarter → Region, more detail)
Just add more fields to `index`.

In [ ]:
drill = pd.pivot_table(df, index=['Year','Quarter','Region'], columns='Category',
                       values='Sales', aggfunc='sum', fill_value=0)
drill

Category             Furniture  Office  Technology
Year Quarter Region                               
2023 Q1      East          531       0         866
             West          296     719           0
     Q2      East          644       0         292
             West          779     845           0
     Q3      East          790       0         426
             West          496     320           0
     Q4      East          898       0         784
             West          299     777           0
2024 Q1      East          708       0         521
             West          570     449           0
     Q2      East          737       0         659
             West          320     550           0
     Q3      East          631       0         279
             West          521     808           0
     Q4      East          270       0         685
             West          262     862           0

## 4. Slice (fix ONE dimension, e.g. Region = East)

In [ ]:
sl = df[df['Region'] == 'East']
pd.pivot_table(sl, index='Quarter', columns='Category', values='Sales', aggfunc='sum', fill_value=0)

Category,Furniture,Technology
Quarter,,
Q1,1239,1387
Q2,1381,951
Q3,1421,705
Q4,1168,1469


## 5. Dice (fix TWO OR MORE dimensions → sub-cube)

In [ ]:
dc = df[(df['Region'].isin(['East','West'])) & (df['Category'].isin(['Furniture','Technology'])) & (df['Year'] == 2024)]
pd.pivot_table(dc, index='Quarter', columns=['Region','Category'], values='Sales', aggfunc='sum', fill_value=0)

Region        East                 West
Category Furniture Technology Furniture
Quarter                                
Q1             708        521       570
Q2             737        659       320
Q3             631        279       521
Q4             270        685       262

## 6. Pivot / Rotate (swap rows and columns to see another view)

In [ ]:
view1 = pd.pivot_table(df, index='Region', columns='Category', values='Sales', aggfunc='sum', fill_value=0)
view2 = view1.T            # rotated: Category on rows, Region on columns
print(view1, '\n'); print(view2)

Category  Furniture  Office  Technology
Region                                 
East           5209       0        4512
West           3543    5330           0 

Region      East  West
Category              
Furniture   5209  3543
Office         0  5330
Technology  4512     0


## 7. Several measures / several aggregate functions

In [ ]:
pd.pivot_table(df, index='Region', values=['Sales','Profit','Quantity'],
               aggfunc={'Sales':'sum', 'Profit':'mean', 'Quantity':'sum'})

,Profit,Quantity,Sales
Region,,,
East,97.7500,92,9721
West,77.3125,81,8873


In [ ]:
pd.pivot_table(df, index='Category', values='Sales', aggfunc=['sum','mean','count','max','min'])

,sum,mean,count,max,min
,Sales,Sales,Sales,Sales,Sales
Category,,,,,
Furniture,8752,547.00,16,898,262
Office,5330,666.25,8,862,320
Technology,4512,564.00,8,866,279


## 8. Quick chart of a pivot

In [ ]:
rollup.drop('Total').drop(columns='Total').plot(kind='bar', figsize=(7,4), title='Sales by Year and Category')
plt.ylabel('Sales'); plt.show()

## Cheat-sheet (write this in the exam if asked)
| Operation | Meaning | Pandas |
|---|---|---|
| Roll-up | less detail | fewer fields in `index` |
| Drill-down | more detail | more fields in `index` |
| Slice | fix 1 dimension | `df[df.Region=='East']` |
| Dice | fix 2+ dimensions | `df[df.Region.isin([...]) & df.Category.isin([...])]` |
| Pivot | rotate axes | swap `index` and `columns`, or `.T` |